In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import time
import tracemalloc
import numpy as np
import sys
import os

In [3]:
sys.path.append(os.path.abspath('..'))
from moogp.model import MOOGP, init_phi, build_cross_K, make_G, build_Gy, vecF, unvecF
from moogp.kernels import make_c_star_matrix, make_c_star_diag

In [4]:
def benchmark_predict_variance():
    print("=== Setting up Model & Data ===")
    rng = np.random.default_rng(42)
    
    # Training Data (Keep this moderate)
    n, d, p, q = 300, 5, 20, 5  
    X = rng.uniform(-1.0, 1.0, size=(n, d))
    Y = rng.normal(size=(n, p))
    data = {"X_scaled": X, "Y": Y}
    
    # Test Data (Make this MASSIVE to stress-test the memory)
    n_test = 5000  
    X_test = rng.uniform(-1.0, 1.0, size=(n_test, d))
    print(f"Training points: {n}, Test points: {n_test}, Outputs: {p}")

    # Build and fit a fast model
    sigma_eps2 = np.full(p, 0.1)
    from scipy.linalg import svd
    _, svals, Vt = svd(Y, full_matrices=False)
    Phi = Vt.T[:, :q] * np.sqrt(n) / svals[:q]
    Psi_fair = np.diag(np.sqrt(sigma_eps2)) @ Phi

    theta0 = np.zeros(q * (d + 1))
    model = MOOGP(terms=[None], q=q, Psi=Psi_fair, learn_Psi=False, 
                  sigma_eps2=sigma_eps2, use_diagonalized_interaction=True)
    model.fit(data, theta0, optimizer_opts={"maxiter": 2}) # 2 iters just to populate cache
    
    # Extract cache variables for the test
    lat_params = model.cache["lat_params"]
    Psi = model.cache["Psi"]
    terms = model.cache["terms"]
    one_based = model.cache["one_based"]

    print("\n=== Starting Benchmark ===")

    # ---------------------------------------------------------
    # TEST 1: The OLD Way (Building the full dense test matrix)
    # ---------------------------------------------------------
    print("\n[OLD METHOD] Dense Matrix Allocation...")
    tracemalloc.start()
    start_time = time.perf_counter()

    diag_prior_old = np.zeros(n_test * p)
    for j, (sigma2_j, ell_j) in enumerate(lat_params):
        
        Cj2_dense = make_c_star_matrix(X_test, X_test, ell=ell_j, sigma2=sigma2_j, terms=terms)
        diag_prior_old += np.kron(Psi[:, j]**2, np.diag(Cj2_dense))

    time_old = time.perf_counter() - start_time
    _, peak_mem_old = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    
    peak_mb_old = peak_mem_old / (1024 * 1024)
    print(f"Time:   {time_old:.4f} seconds")
    print(f"Memory: {peak_mb_old:.2f} MB")

    # ---------------------------------------------------------
    # TEST 2: The NEW Way (Analytical Diagonal only)
    # ---------------------------------------------------------
    print("\n[NEW METHOD] Analytical Diagonal...")
    tracemalloc.start()
    start_time = time.perf_counter()

    diag_prior_new = np.zeros(n_test * p)
    for j, (sigma2_j, ell_j) in enumerate(lat_params):
        
        Cj2_diag = make_c_star_diag(X_test, ell=ell_j, sigma2=sigma2_j, terms=terms)
        diag_prior_new += np.kron(Psi[:, j]**2, Cj2_diag)

    time_new = time.perf_counter() - start_time
    _, peak_mem_new = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    peak_mb_new = peak_mem_new / (1024 * 1024)
    print(f"Time:   {time_new:.4f} seconds")
    print(f"Memory: {peak_mb_new:.2f} MB")

    # ---------------------------------------------------------
    # Verify Math and Results
    # ---------------------------------------------------------
    print("\n=== Results ===")
    assert np.allclose(diag_prior_old, diag_prior_new), "Math mismatch! The two methods returned different arrays."
    
    mem_saved = peak_mb_old / peak_mb_new if peak_mb_new > 0 else float('inf')
    time_saved = time_old / time_new if time_new > 0 else float('inf')
    
    print(f"Math is identical: TRUE")
    print(f"Speedup:           {time_saved:.1f}x Faster")
    print(f"Memory Efficiency: {mem_saved:.1f}x Less RAM used")

benchmark_predict_variance()

=== Setting up Model & Data ===
Training points: 300, Test points: 5000, Outputs: 20

=== Starting Benchmark ===

[OLD METHOD] Dense Matrix Allocation...
Time:   7.5299 seconds
Memory: 2289.65 MB

[NEW METHOD] Analytical Diagonal...
Time:   0.0131 seconds
Memory: 1.69 MB

=== Results ===
Math is identical: TRUE
Speedup:           575.8x Faster
Memory Efficiency: 1353.7x Less RAM used
